# Layer Functional Analysis

Characterize what each layer does: information gain, transformation
magnitude, specialization, and prediction contribution.

## Why This Matters

Layer functional characterizes what each layer contributes to the model's computation. Understanding layer-level organization — which layers are critical, which are redundant, and how they specialize — is essential for both interpretability and efficient model design.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layer_functional_analysis import (
    layer_information_gain, layer_transformation_magnitude,
    layer_specialization_score, layer_prediction_contribution,
    layer_functional_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Information Gain

How much does each layer reduce prediction entropy?

In [ ]:
result = layer_information_gain(model, tokens, position=-1)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: H={p['entropy']:.4f} KL={p['kl_from_previous']:.4f}")

## Transformation Magnitude

How much does each layer change the residual stream?

In [ ]:
result = layer_transformation_magnitude(model, tokens)
for p in result['per_layer']:
    bar = '█' * int(p['delta_norm'] * 10)
    print(f"  Layer {p['layer']}: delta={p['delta_norm']:.4f} "
          f"relative={p['relative_change']:.4f} "
          f"cos={p['pre_post_cosine']:.4f} {bar}")

## Specialization

Is each layer attention-dominant, MLP-dominant, or balanced?

In [ ]:
result = layer_specialization_score(model, tokens)
for p in result['per_layer']:
    attn_bar = '█' * int(p['attn_fraction'] * 20)
    mlp_bar = '█' * int(p['mlp_fraction'] * 20)
    print(f"  Layer {p['layer']}: [{p['specialization']:14s}] "
          f"attn={p['attn_fraction']:.3f} {attn_bar}")
    print(f"               mlp={p['mlp_fraction']:.3f} {mlp_bar}")

## Prediction Contribution

Which layers change the top prediction?

In [ ]:
result = layer_prediction_contribution(model, tokens, position=-1)
for p in result['per_layer']:
    flag = ' [CHANGED]' if p['changed'] else ''
    print(f"  Layer {p['layer']}: {p['pre_prediction']} -> {p['post_prediction']} "
          f"delta={p['logit_delta_norm']:.4f}{flag}")

## Functional Summary

In [ ]:
result = layer_functional_summary(model, tokens, position=-1)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: H={p['entropy']:.4f} "
          f"delta={p['delta_norm']:.4f} "
          f"{p['specialization']} (bal={p['balance']:.3f})")